# DS 5020 – Mini-Project 02: Will This Be a Strong Sales Week?
**Complete worked solution | Bayes’ theorem & Naive Bayes on Walmart.csv**

Place `Walmart.csv` in the same folder as this notebook, then select **Kernel → Restart & Run All**. Run the cells in order. Parts P/B use the full-dataset store average. Part N uses the 2010–2011 **training-only** average to avoid leakage.

**Definitions (R1):** a store-week is one row in the dataset (R2). The seven events are:

|Event|Meaning|
|---|---|
|S|Weekly_Sales ≥ 1.10 × that store's average weekly sales|
|H|Holiday_Flag = 1|
|Q|Date month is November or December|
|C|Temperature < 40°F|
|F|Fuel_Price ≥ $3.50|
|J|Unemployment ≥ 8%|
|G|Store average weekly sales ≥ $1,000,000|

**R5:** Check probabilities by a complement, by counts, by summing to one, or by a separate formula. **R4:** Use store numbers only, no identities/addresses.

## Warm-up W1–W7 (by hand, using the 20-row table on p. 3)
The table has 20 store-weeks: 6 strong and 14 not strong. There are 4 holiday weeks, 5 November–December weeks, 3 strong holiday weeks, and 4 strong November–December weeks.

**W1. Marginal probabilities:**

- \(P(S)=6/20=0.30\).
- \(P(H)=4/20=0.20\).
- \(P(Q)=5/20=0.25\).

**W2. Conditionals:** \(P(S\mid H)=3/4=0.75\), and \(P(S\mid H^c)=3/16=0.1875\). A strong week is **4 times as likely** on a holiday week: \(0.75/0.1875=4\).

**W3. Bayes:** \(P(H\mid S)=\frac{P(S\mid H)P(H)}{P(S)}=\frac{0.75(0.20)}{0.30}=0.50\). Check: 3 of the 6 strong weeks are holidays, \(3/6=0.50\).

**W4. Total probability:** \(P(S)=P(S\mid H)P(H)+P(S\mid H^c)P(H^c)=0.75(0.20)+0.1875(0.80)=0.30\), matching W1.

**W5. Independence:** \(P(S\cap H)=3/20=0.15\), whereas \(P(S)P(H)=0.30(0.20)=0.06\). They are **not independent**.

**W6. Holiday staffing rule:** precision \(=P(S\mid H)=0.75\); recall \(=P(H\mid S)=0.50\); fraction of strong weeks missed \(=1-0.50=0.50\) (**50%**).

**W7. Naive Bayes by hand (no smoothing for the small table):**

For a week with **H=1, Q=1**:

- Strong score = \(P(S)P(H\mid S)P(Q\mid S)=(6/20)(3/6)(4/6)=0.100000\).
- Not-strong score = \(P(S^c)P(H\mid S^c)P(Q\mid S^c)=(14/20)(1/14)(1/14)=0.00357143\).
- Normalized \(P(S\mid H,Q)\approx0.100000/(0.100000+0.00357143)=\mathbf{0.9655}\). True observed fraction in that table column: \(2/2=1.0000\).

For a week with **H=0, Q=0**:

- Strong score = \((6/20)(3/6)(2/6)=0.050000\).
- Not-strong score = \((14/20)(13/14)(13/14)=0.60357143\).
- Normalized \(P(S\mid H^c,Q^c)\approx0.050000/(0.050000+0.60357143)=\mathbf{0.0765}\). True observed fraction: \(1/13\approx0.0769\).

**Interpretation:** Naive Bayes approximately matches the nonholiday/nonseason case; for holiday+season it predicts ~96.55% versus 100% in the small sample. These are not necessarily identical because the independence assumption is approximate.

## Cell 1 — Imports and P1: load/check the data

Import the libraries, read the file, parse the date as day–month–year, and check the dataset against the assignment.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

CSV_PATH = Path("Walmart.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("/mnt/data/Walmart.csv")  # available in this ChatGPT workspace

df = pd.read_csv(CSV_PATH)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

print("Rows:", len(df))
print("Stores:", df["Store"].nunique())
print("Unique weeks:", df["Date"].nunique())
print("Holiday store-weeks:", df["Holiday_Flag"].sum())
print("Missing values:", df.isna().sum().sum())
display(df.head())

assert len(df) == 6435
assert df["Store"].nunique() == 45
assert df["Date"].nunique() == 143
assert df["Holiday_Flag"].sum() == 450
assert df.isna().sum().sum() == 0

Rows: 6435
Stores: 45
Unique weeks: 143
Holiday store-weeks: 450
Missing values: 0


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106


## Cell 2 — Define all seven events (R1)

Create a reusable function. For P/B use every store’s overall mean; later, N1 calls this function with training-only means.

In [2]:
def add_events(data, store_avg):
    data = data.copy()
    avg = data["Store"].map(store_avg)
    assert avg.notna().all(), "All stores must have a training average."
    data["S"] = data["Weekly_Sales"] >= 1.10 * avg
    data["H"] = data["Holiday_Flag"] == 1
    data["Q"] = data["Date"].dt.month.isin([11, 12])
    data["C"] = data["Temperature"] < 40
    data["F"] = data["Fuel_Price"] >= 3.50
    data["J"] = data["Unemployment"] >= 8
    data["G"] = avg >= 1_000_000
    return data

full_store_avg = df.groupby("Store")["Weekly_Sales"].mean()
ev = add_events(df, full_store_avg)
EVENTS = ["S", "H", "Q", "C", "F", "J", "G"]
FEATURES = ["H", "Q", "C", "F", "J", "G"]
display(ev[["Store", "Date", "Weekly_Sales"] + EVENTS].head())

,Store,Date,Weekly_Sales,S,H,Q,C,F,J,G
0,1,2010-02-05,1643690.90,False,False,False,False,False,True,True
1,1,2010-02-12,1641957.44,False,True,False,True,False,True,True
2,1,2010-02-19,1611968.17,False,False,False,True,False,True,True
3,1,2010-02-26,1409727.59,False,False,False,False,False,True,True
4,1,2010-03-05,1554806.68,False,False,False,False,False,True,True


## Cell 3 — P2: event probabilities with complement checks

For each event, calculate count/6435 and independently confirm P(E) + P(not E) = 1.

In [3]:
p2_rows = []
for event in EVENTS:
    count = int(ev[event].sum())
    p_count = count / len(ev)
    p_complement = (~ev[event]).sum() / len(ev)
    p2_rows.append({
        "Event": event, "Count": count, "P(Event)": p_count,
        "P(Not event)": p_complement, "Sum (check=1)": p_count + p_complement
    })
p2 = pd.DataFrame(p2_rows)
display(p2.round(6))
assert np.allclose(p2["Sum (check=1)"], 1.0)

,Event,Count,P(Event),P(Not event),Sum (check=1)
0,S,814,0.126496,0.873504,1.0
1,H,450,0.069930,0.930070,1.0
2,Q,810,0.125874,0.874126,1.0
3,C,989,0.153691,0.846309,1.0
4,F,3029,0.470707,0.529293,1.0
5,J,3022,0.469619,0.530381,1.0
6,G,3146,0.488889,0.511111,1.0


## Cell 4 — P3: joint probabilities

For each pair, count rows satisfying both events; check against the mean of the Boolean intersection.

In [4]:
pairs_p3 = [("S", "H"), ("S", "Q"), ("H", "Q"), ("S", "C")]
p3_rows = []
for a, b in pairs_p3:
    together = ev[a] & ev[b]
    p_count = together.sum() / len(ev)
    p_mean = together.mean()
    assert np.isclose(p_count, p_mean)
    p3_rows.append({"Joint event": f"P({a} ∩ {b})", "Count": int(together.sum()),
                    "Probability": p_count, "Second check": p_mean})
p3 = pd.DataFrame(p3_rows)
display(p3.round(6))

,Joint event,Count,Probability,Second check
0,P(S ∩ H),122,0.018959,0.018959
1,P(S ∩ Q),333,0.051748,0.051748
2,P(H ∩ Q),180,0.027972,0.027972
3,P(S ∩ C),185,0.028749,0.028749


## Cell 5 — B1: strong sales conditional on holiday/nonholiday

Compute P(S|H) and P(S|Hᶜ) using counts, check with group means, and compare.

In [5]:
strong_h = int((ev["S"] & ev["H"]).sum())
n_h = int(ev["H"].sum())
strong_nonh = int((ev["S"] & ~ev["H"]).sum())
n_nonh = int((~ev["H"]).sum())
p_s_given_h = strong_h / n_h
p_s_given_nonh = strong_nonh / n_nonh
assert np.isclose(p_s_given_h, ev.loc[ev["H"], "S"].mean())
assert np.isclose(p_s_given_nonh, ev.loc[~ev["H"], "S"].mean())
print(f"P(S|H) = {strong_h}/{n_h} = {p_s_given_h:.6f} ({100*p_s_given_h:.2f}%)")
print(f"P(S|Hᶜ) = {strong_nonh}/{n_nonh} = {p_s_given_nonh:.6f} ({100*p_s_given_nonh:.2f}%)")
print(f"A strong week is {p_s_given_h/p_s_given_nonh:.2f} times as likely on a holiday week.")

P(S|H) = 122/450 = 0.271111 (27.11%)
P(S|Hᶜ) = 692/5985 = 0.115622 (11.56%)
A strong week is 2.34 times as likely on a holiday week.


## Cell 6 — B2: Bayes’ theorem and counting check

Use P(H|S) = P(S|H)P(H)/P(S), then compute the same probability directly among strong rows.

In [6]:
p_s = ev["S"].mean()
p_h = ev["H"].mean()
p_h_given_s_bayes = p_s_given_h * p_h / p_s
p_h_given_s_count = (ev["H"] & ev["S"]).sum() / ev["S"].sum()
print(f"P(H|S) by Bayes = ({p_s_given_h:.6f} × {p_h:.6f})/{p_s:.6f} = {p_h_given_s_bayes:.6f}")
print(f"P(H|S) by counting = {int((ev['H'] & ev['S']).sum())}/{int(ev['S'].sum())} = {p_h_given_s_count:.6f}")
assert np.isclose(p_h_given_s_bayes, p_h_given_s_count)

P(H|S) by Bayes = (0.271111 × 0.069930)/0.126496 = 0.149877
P(H|S) by counting = 122/814 = 0.149877


## Cell 7 — B3: total probability across five holiday groups

Assign the specific holiday dates listed in the PDF; calculate P(group), P(S|group), contribution, and P(group|S).

In [7]:
holiday_dates = {
    "Super Bowl": ["2010-02-12", "2011-02-11", "2012-02-10"],
    "Labor Day": ["2010-09-10", "2011-09-09", "2012-09-07"],
    "Thanksgiving": ["2010-11-26", "2011-11-25"],
    "Christmas": ["2010-12-31", "2011-12-30"],
}
ev["Holiday_Group"] = "No holiday"
for group, dates in holiday_dates.items():
    ev.loc[ev["Date"].isin(pd.to_datetime(dates)), "Holiday_Group"] = group
assert (ev.loc[ev["H"], "Holiday_Group"] != "No holiday").all()
assert (ev.loc[~ev["H"], "Holiday_Group"] == "No holiday").all()

b3_rows = []
for group in ["Super Bowl", "Labor Day", "Thanksgiving", "Christmas", "No holiday"]:
    sub = ev.loc[ev["Holiday_Group"] == group]
    pg = len(sub) / len(ev)
    psg = sub["S"].mean()
    b3_rows.append({
        "Group": group, "Store-weeks": len(sub), "Strong weeks": int(sub["S"].sum()),
        "P(group)": pg, "P(S|group)": psg,
        "P(S and group)": pg * psg,
        "P(group|S)": int(sub["S"].sum()) / int(ev["S"].sum())
    })
b3 = pd.DataFrame(b3_rows)
display(b3.round(6))
p_s_rebuilt = b3["P(S and group)"].sum()
print(f"P(S) rebuilt = {p_s_rebuilt:.6f}; directly = {p_s:.6f}")
print("Highest P(S|group):", b3.loc[b3["P(S|group)"].idxmax(), "Group"])
print("Largest share P(group|S):", b3.loc[b3["P(group|S)"].idxmax(), "Group"])
assert np.isclose(p_s_rebuilt, p_s)
assert np.isclose(b3["P(group)"].sum(), 1)
assert np.isclose(b3["P(group|S)"].sum(), 1)

,Group,Store-weeks,Strong weeks,P(group),P(S|group),P(S and group),P(group|S)
0,Super Bowl,135,26,0.020979,0.192593,0.004040,0.031941
1,Labor Day,135,16,0.020979,0.118519,0.002486,0.019656
2,Thanksgiving,90,74,0.013986,0.822222,0.011500,0.090909
3,Christmas,90,6,0.013986,0.066667,0.000932,0.007371
4,No holiday,5985,692,0.930070,0.115622,0.107537,0.850123


P(S) rebuilt = 0.126496; directly = 0.126496
Highest P(S|group): Thanksgiving
Largest share P(group|S): No holiday


## Cell 8 — B4: November–December conditionals

The denominators differ: P(S|Q) conditions on November–December; P(Q|S) conditions on strong weeks.

In [8]:
n_s_and_q = int((ev["S"] & ev["Q"]).sum())
p_s_given_q = n_s_and_q / int(ev["Q"].sum())
p_q_given_s = n_s_and_q / int(ev["S"].sum())
assert np.isclose(p_s_given_q, ev.loc[ev["Q"], "S"].mean())
assert np.isclose(p_q_given_s, ev.loc[ev["S"], "Q"].mean())
print(f"P(S|Q) = {n_s_and_q}/{int(ev['Q'].sum())} = {p_s_given_q:.6f}")
print(f"P(Q|S) = {n_s_and_q}/{int(ev['S'].sum())} = {p_q_given_s:.6f}")
print("They differ because they use different reference groups (different denominators).")

P(S|Q) = 333/810 = 0.411111
P(Q|S) = 333/814 = 0.409091
They differ because they use different reference groups (different denominators).


## Cell 9 — B5: holiday vs November–December staffing rules

Compute each rule’s precision with the reverse Bayes formula; check precision and recall by counting.

In [9]:
def staffing_rule(event):
    p_e_given_s = ev.loc[ev["S"], event].mean()
    p_e_given_not_s = ev.loc[~ev["S"], event].mean()
    precision_bayes = (p_e_given_s * p_s) / (
        p_e_given_s * p_s + p_e_given_not_s * (1 - p_s)
    )
    precision_count = ev.loc[ev[event], "S"].mean()
    recall_count = ev.loc[ev["S"], event].mean()
    assert np.isclose(precision_bayes, precision_count)
    return {
        "Rule": "Holiday only" if event == "H" else "November–December",
        "Precision": precision_bayes,
        "Recall": recall_count,
        "Strong weeks missed": 1 - recall_count,
        "Flagged store-weeks": int(ev[event].sum())
    }
b5 = pd.DataFrame([staffing_rule("H"), staffing_rule("Q")])
display(b5.round(6))
print("A larger recall means the rule misses fewer truly strong weeks.")

,Rule,Precision,Recall,Strong weeks missed,Flagged store-weeks
0,Holiday only,0.271111,0.149877,0.850123,450
1,November–December,0.411111,0.409091,0.590909,810


A larger recall means the rule misses fewer truly strong weeks.


## Cell 10 — B6: hypothetical base rates

Hold the holiday likelihoods P(H|S) and P(H|Sᶜ) fixed, but change the prior P(S).

In [10]:
p_h_given_not_s = ev.loc[~ev["S"], "H"].mean()
def holiday_precision_given_prior(new_prior):
    return (p_h_given_s_count * new_prior) / (
        p_h_given_s_count * new_prior + p_h_given_not_s * (1 - new_prior)
    )
b6 = pd.DataFrame([
    {"Scenario": "Twice as common", "Assumed P(S)": 2 * p_s,
     "Holiday precision": holiday_precision_given_prior(2 * p_s)},
    {"Scenario": "Half as common", "Assumed P(S)": p_s / 2,
     "Holiday precision": holiday_precision_given_prior(p_s / 2)},
])
display(b6.round(6))
assert 2 * p_s < 1

,Scenario,Assumed P(S),Holiday precision
0,Twice as common,0.252991,0.465204
1,Half as common,0.063248,0.147789


## Cell 11 — B7: independence checks

Compare P(S ∩ E) against P(S)P(E); a nonzero difference means dependence in this dataset.

In [11]:
b7_rows = []
for feature in FEATURES:
    joint = (ev["S"] & ev[feature]).mean()
    if_independent = ev["S"].mean() * ev[feature].mean()
    b7_rows.append({
        "Pair": f"S and {feature}", "P(S∩E)": joint,
        "P(S)P(E)": if_independent, "Difference": joint - if_independent,
        "Abs. difference": abs(joint - if_independent)
    })
b7 = pd.DataFrame(b7_rows)
display(b7.round(6))
print("Largest absolute departures from independence:")
display(b7.sort_values("Abs. difference", ascending=False)[["Pair", "Difference"]].round(6))

,Pair,P(S∩E),P(S)P(E),Difference,Abs. difference
0,S and H,0.018959,0.008846,0.010113,0.010113
1,S and Q,0.051748,0.015923,0.035826,0.035826
2,S and C,0.028749,0.019441,0.009308,0.009308
3,S and F,0.045221,0.059542,-0.014321,0.014321
4,S and J,0.064025,0.059405,0.004620,0.004620
5,S and G,0.057032,0.061842,-0.004810,0.004810


Largest absolute departures from independence:


,Pair,Difference
1,S and Q,0.035826
3,S and F,-0.014321
0,S and H,0.010113
2,S and C,0.009308
5,S and G,-0.004810
4,S and J,0.004620


## Cell 12 — N1: time-based train/test split (prevent leakage)

Train on 2010–2011 (4,500 store-weeks), test on 2012 (1,935). Recompute store averages using only train weeks.

In [12]:
train_raw = df.loc[df["Date"].dt.year.isin([2010, 2011])].copy()
test_raw = df.loc[df["Date"].dt.year == 2012].copy()
train_store_avg = train_raw.groupby("Store")["Weekly_Sales"].mean()
train = add_events(train_raw, train_store_avg)
test = add_events(test_raw, train_store_avg)
print(f"Training: {len(train)} rows; P(S) = {train['S'].mean():.6f} ({100*train['S'].mean():.2f}%)")
print(f"Test:     {len(test)} rows; P(S) = {test['S'].mean():.6f} ({100*test['S'].mean():.2f}%)")
assert len(train) == 4500 and len(test) == 1935
assert train["Date"].max() < test["Date"].min()
assert train["Store"].nunique() == test["Store"].nunique() == 45

Training: 4500 rows; P(S) = 0.127556 (12.76%)
Test:     1935 rows; P(S) = 0.131783 (13.18%)


## Cell 13 — N2: priors and six binary likelihood tables

First show frequencies estimated by counting. Also show add-one (Laplace) smoothed probabilities (alpha=1) to match CategoricalNB in N5.

In [13]:
n_train = len(train)
n_s = int(train["S"].sum())
n_not_s = n_train - n_s
prior_s = n_s / n_train
prior_not_s = n_not_s / n_train
print(f"P(S)={n_s}/{n_train}={prior_s:.6f}")
print(f"P(Sᶜ)={n_not_s}/{n_train}={prior_not_s:.6f}")
assert np.isclose(prior_s + prior_not_s, 1)

n2_rows = []
for feature in FEATURES:
    for value in (0, 1):
        yes_count = int(((train[feature].astype(int) == value) & train["S"]).sum())
        no_count = int(((train[feature].astype(int) == value) & ~train["S"]).sum())
        n2_rows.append({
            "Feature": feature, "Value": value,
            "Count if S": yes_count, "Count if Sᶜ": no_count,
            "P(value|S) raw": yes_count / n_s,
            "P(value|Sᶜ) raw": no_count / n_not_s,
            "P(value|S) smoothed": (yes_count + 1) / (n_s + 2),
            "P(value|Sᶜ) smoothed": (no_count + 1) / (n_not_s + 2),
        })
n2 = pd.DataFrame(n2_rows)
display(n2.round(6))
for feature in FEATURES:
    sub = n2.loc[n2["Feature"] == feature]
    for column in ["P(value|S) raw", "P(value|Sᶜ) raw",
                   "P(value|S) smoothed", "P(value|Sᶜ) smoothed"]:
        assert np.isclose(sub[column].sum(), 1)

P(S)=574/4500=0.127556
P(Sᶜ)=3926/4500=0.872444


,Feature,Value,Count if S,Count if Sᶜ,P(value|S) raw,P(value|Sᶜ) raw,P(value|S) smoothed,P(value|Sᶜ) smoothed
0,H,0,475,3665,0.827526,0.933520,0.826389,0.933299
1,H,1,99,261,0.172474,0.066480,0.173611,0.066701
2,Q,0,239,3451,0.416376,0.879012,0.416667,0.878819
3,Q,1,335,475,0.583624,0.120988,0.583333,0.121181
4,C,0,409,3327,0.712544,0.847427,0.711806,0.847251
5,C,1,165,599,0.287456,0.152573,0.288194,0.152749
6,F,0,466,2568,0.811847,0.654101,0.810764,0.654022
7,F,1,108,1358,0.188153,0.345899,0.189236,0.345978
8,J,0,257,1841,0.447735,0.468925,0.447917,0.468941
9,J,1,317,2085,0.552265,0.531075,0.552083,0.531059


## Cell 14 — N3: manual Naive Bayes scores and normalization

Profile (a): holiday, Nov–Dec, cold, cheap fuel, low unemployment, large store. Profile (b) is the opposite. Use the smoothed conditional probabilities from N2.

In [14]:
profile_a = {"H": 1, "Q": 1, "C": 1, "F": 0, "J": 0, "G": 1}
profile_b = {"H": 0, "Q": 0, "C": 0, "F": 1, "J": 1, "G": 0}

def manual_nb_posterior(profile, label):
    score_s = prior_s
    score_not_s = prior_not_s
    steps = []
    for feature, value in profile.items():
        p_x_given_s = (int(((train[feature].astype(int) == value) & train["S"]).sum()) + 1) / (n_s + 2)
        p_x_given_not_s = (int(((train[feature].astype(int) == value) & ~train["S"]).sum()) + 1) / (n_not_s + 2)
        score_s *= p_x_given_s
        score_not_s *= p_x_given_not_s
        steps.append({"Feature": feature, "Value": value,
                      "P(value|S) smoothed": p_x_given_s,
                      "P(value|Sᶜ) smoothed": p_x_given_not_s})
    posterior = score_s / (score_s + score_not_s)
    other_posterior = score_not_s / (score_s + score_not_s)
    print(f"\nProfile {label}: {profile}")
    display(pd.DataFrame(steps).round(6))
    print(f"Score(S)  = P(S) × ∏P(feature|S)   = {score_s:.12f}")
    print(f"Score(Sᶜ) = P(Sᶜ) × ∏P(feature|Sᶜ) = {score_not_s:.12f}")
    print(f"P(S|x) = {score_s:.12f}/({score_s:.12f}+{score_not_s:.12f}) = {posterior:.9f}")
    assert np.isclose(posterior + other_posterior, 1)
    return posterior

posterior_a = manual_nb_posterior(profile_a, "a")
posterior_b = manual_nb_posterior(profile_b, "b")


Profile a: {'H': 1, 'Q': 1, 'C': 1, 'F': 0, 'J': 0, 'G': 1}


,Feature,Value,P(value|S) smoothed,P(value|Sᶜ) smoothed
0,H,1,0.173611,0.066701
1,Q,1,0.583333,0.121181
2,C,1,0.288194,0.152749
3,F,0,0.810764,0.654022
4,J,0,0.447917,0.468941
5,G,1,0.463542,0.492617


Score(S)  = P(S) × ∏P(feature|S)   = 0.000626699995
Score(Sᶜ) = P(Sᶜ) × ∏P(feature|Sᶜ) = 0.000162743293
P(S|x) = 0.000626699995/(0.000626699995+0.000162743293) = 0.793850558

Profile b: {'H': 0, 'Q': 0, 'C': 0, 'F': 1, 'J': 1, 'G': 0}


,Feature,Value,P(value|S) smoothed,P(value|Sᶜ) smoothed
0,H,0,0.826389,0.933299
1,Q,0,0.416667,0.878819
2,C,0,0.711806,0.847251
3,F,1,0.189236,0.345978
4,J,1,0.552083,0.531059
5,G,0,0.536458,0.507383


Score(S)  = P(S) × ∏P(feature|S)   = 0.001752179628
Score(Sᶜ) = P(Sᶜ) × ∏P(feature|Sᶜ) = 0.056519268402
P(S|x) = 0.001752179628/(0.001752179628+0.056519268402) = 0.030069265


## Cell 15 — N4: check conditional independence of H and Q

Test whether P(H ∩ Q|S) equals P(H|S) × P(Q|S) in the training data.

In [15]:
strong_train = train.loc[train["S"]]
p_hq_given_s = (strong_train["H"] & strong_train["Q"]).mean()
p_h_s_times_p_q_s = strong_train["H"].mean() * strong_train["Q"].mean()
print(f"P(H∩Q|S) = {p_hq_given_s:.6f}")
print(f"P(H|S)P(Q|S) = {p_h_s_times_p_q_s:.6f}")
print(f"Difference = {p_hq_given_s - p_h_s_times_p_q_s:.6f}")
print("H and Q are not independent: Thanksgiving and Christmas holidays fall in Nov–Dec.")

P(H∩Q|S) = 0.139373
P(H|S)P(Q|S) = 0.100660
Difference = 0.038713
H and Q are not independent: Thanksgiving and Christmas holidays fall in Nov–Dec.


## Cell 16 — N5: fit CategoricalNB and verify N3

Use exactly the same six binary features and alpha=1 Laplace smoothing as in N3.

In [16]:
X_train = train[FEATURES].astype(int)
y_train = train["S"].astype(int)
X_test = test[FEATURES].astype(int)
y_test = test["S"].astype(int)

nb = CategoricalNB(alpha=1)
nb.fit(X_train, y_train)
profiles_df = pd.DataFrame([profile_a, profile_b], index=["Profile a", "Profile b"])[FEATURES]
nb_posteriors = nb.predict_proba(profiles_df)[:, list(nb.classes_).index(1)]
comparison_n5 = pd.DataFrame({"Profile": profiles_df.index,
                              "Manual P(S|x)": [posterior_a, posterior_b],
                              "sklearn P(S|x)": nb_posteriors})
display(comparison_n5.round(9))
assert np.allclose([posterior_a, posterior_b], nb_posteriors, atol=1e-12)
print("The manual calculations match scikit-learn.")

,Profile,Manual P(S|x),sklearn P(S|x)
0,Profile a,0.793851,0.793851
1,Profile b,0.030069,0.030069


The manual calculations match scikit-learn.


## Cell 17 — N6: confusion matrices and baseline comparisons

Evaluate on 2012 only: never strong, holiday-only rule, and Naive Bayes with the default 0.5 decision threshold.

In [17]:
def evaluation_row(name, actual, predicted):
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
    precision_count = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_count = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    assert np.isclose(precision_count, precision_score(actual, predicted, zero_division=0))
    assert np.isclose(recall_count, recall_score(actual, predicted, zero_division=0))
    return {"Method": name, "TN": tn, "FP": fp, "FN": fn, "TP": tp,
            "Accuracy": accuracy_score(actual, predicted),
            "Precision": precision_count, "Recall": recall_count}

pred_never = np.zeros(len(test), dtype=int)
pred_holiday = test["H"].astype(int).to_numpy()
pred_nb = nb.predict(X_test)
n6 = pd.DataFrame([
    evaluation_row("Never strong", y_test, pred_never),
    evaluation_row("Holiday only", y_test, pred_holiday),
    evaluation_row("Naive Bayes (0.5)", y_test, pred_nb),
])
display(n6.round(6))
for name, predicted in [("Never strong", pred_never), ("Holiday only", pred_holiday),
                        ("Naive Bayes", pred_nb)]:
    print(f"\n{name} confusion matrix [[TN,FP],[FN,TP]]:")
    print(confusion_matrix(y_test, predicted, labels=[0, 1]))
print("\n2012 has no Thanksgiving or Christmas records (data ends in October).")
print("This limits evaluation of holiday-season effects learned from 2010–2011.")

,Method,TN,FP,FN,TP,Accuracy,Precision,Recall
0,Never strong,1680,0,255,0,0.868217,0.000000,0.000000
1,Holiday only,1613,67,232,23,0.845478,0.255556,0.090196
2,Naive Bayes (0.5),1680,0,255,0,0.868217,0.000000,0.000000



Never strong confusion matrix [[TN,FP],[FN,TP]]:
[[1680    0]
 [ 255    0]]

Holiday only confusion matrix [[TN,FP],[FN,TP]]:
[[1613   67]
 [ 232   23]]

Naive Bayes confusion matrix [[TN,FP],[FN,TP]]:
[[1680    0]
 [ 255    0]]

2012 has no Thanksgiving or Christmas records (data ends in October).
This limits evaluation of holiday-season effects learned from 2010–2011.


## Cell 18 — N7: threshold stretch task

Predict strong when P(S|x) is strictly greater than t; compare four thresholds.

In [18]:
test_proba = nb.predict_proba(X_test)[:, list(nb.classes_).index(1)]
n7_rows = []
for t in [0.1, 0.2, 0.3, 0.5]:
    predicted = (test_proba > t).astype(int)
    row = evaluation_row(f"t={t}", y_test, predicted)
    n7_rows.append({"Threshold": t, "Predicted strong": int(predicted.sum()),
                    "Accuracy": row["Accuracy"], "Precision": row["Precision"],
                    "Recall": row["Recall"]})
n7 = pd.DataFrame(n7_rows)
display(n7.round(6))
print("Lower thresholds usually flag more weeks; precision and recall need not both improve.")

,Threshold,Predicted strong,Accuracy,Precision,Recall
0,0.1,125,0.823256,0.152,0.074510
1,0.2,10,0.866150,0.300,0.011765
2,0.3,2,0.867183,0.000,0.000000
3,0.5,0,0.868217,0.000,0.000000


Lower thresholds usually flag more weeks; precision and recall need not both improve.


## Cell 19 — J1: five-sentence note to Maria (note.txt)

Build the note using measured values. Print and save it as note.txt, as required by the project.

In [19]:
thanksgiving_p = float(b3.loc[b3["Group"] == "Thanksgiving", "P(S|group)"].iloc[0])
holiday_precision = float(b5.loc[b5["Rule"] == "Holiday only", "Precision"].iloc[0])
holiday_recall = float(b5.loc[b5["Rule"] == "Holiday only", "Recall"].iloc[0])
nb_recall = float(n6.loc[n6["Method"] == "Naive Bayes (0.5)", "Recall"].iloc[0])
holiday_test_recall = float(n6.loc[n6["Method"] == "Holiday only", "Recall"].iloc[0])

sentences = [
    f"Across all 6,435 store-weeks, {100*p_s:.2f}% were strong sales weeks.",
    f"Thanksgiving was the strongest holiday ({100*thanksgiving_p:.2f}% strong), while {100*p_s_given_q:.2f}% of November–December store-weeks were strong.",
    f"The holiday-only staffing rule was right {100*holiday_precision:.2f}% of the time it flagged a week, but missed {100*(1-holiday_recall):.2f}% of all strong weeks.",
    f"On the 2012 test set, Naive Bayes at the standard 0.5 cutoff did not beat the holiday rule at finding strong weeks: its recall was {100*nb_recall:.2f}% versus {100*holiday_test_recall:.2f}% for the holiday rule.",
    "One risk is that the 2012 test data end in October, so the model cannot be tested on that year's Thanksgiving or Christmas periods."
]
note_text = " ".join(sentences)
print(note_text)
Path("note.txt").write_text(note_text + "\n", encoding="utf-8")
assert len(sentences) == 5

Across all 6,435 store-weeks, 12.65% were strong sales weeks. Thanksgiving was the strongest holiday (82.22% strong), while 41.11% of November–December store-weeks were strong. The holiday-only staffing rule was right 27.11% of the time it flagged a week, but missed 85.01% of all strong weeks. On the 2012 test set, Naive Bayes at the standard 0.5 cutoff did not beat the holiday rule at finding strong weeks: its recall was 0.00% versus 9.02% for the holiday rule. One risk is that the 2012 test data end in October, so the model cannot be tested on that year's Thanksgiving or Christmas periods.


## Cell 20 — Section 7: filled-in answer sheet

Collect the exact requested values into one table and export a CSV. B7 and N7 detailed tables appear above.

In [20]:
highest_holiday = b3.loc[b3.loc[b3["Group"] != "No holiday", "P(S|group)"].idxmax(), "Group"]
strong_share_group = b3.loc[b3["P(group|S)"].idxmax(), "Group"]

answer_rows = [
    ("P1", f"{len(df)} rows; {df.Store.nunique()} stores; {df.Date.nunique()} weeks; {int(df.Holiday_Flag.sum())} holiday store-weeks"),
    ("P2", "; ".join(f"P({e})={ev[e].mean():.6f}" for e in EVENTS)),
    ("P3", "; ".join(f"P({a}∩{b})={(ev[a]&ev[b]).mean():.6f}" for a, b in pairs_p3)),
    ("B1", f"P(S|H)={p_s_given_h:.6f}; P(S|Hᶜ)={p_s_given_nonh:.6f}"),
    ("B2", f"P(H|S) by Bayes={p_h_given_s_bayes:.6f}; counting={p_h_given_s_count:.6f}"),
    ("B3", f"Highest P(S|group): {highest_holiday}; largest P(group|S): {strong_share_group}"),
    ("B4", f"P(S|Q)={p_s_given_q:.6f}; P(Q|S)={p_q_given_s:.6f}"),
    ("B5", "; ".join(f"{r['Rule']}: precision={r['Precision']:.6f}, recall={r['Recall']:.6f}" for _, r in b5.iterrows())),
    ("B6", "; ".join(f"{r['Scenario']}: {r['Holiday precision']:.6f}" for _, r in b6.iterrows())),
    ("N3", f"Profile a={posterior_a:.9f}; Profile b={posterior_b:.9f}"),
    ("N6", "; ".join(f"{r['Method']}: accuracy={r['Accuracy']:.6f}, precision={r['Precision']:.6f}, recall={r['Recall']:.6f}" for _, r in n6.iterrows())),
]
answer_sheet = pd.DataFrame(answer_rows, columns=["Task", "Your value"])
display(answer_sheet)
answer_sheet.to_csv("DS5020_Walmart_Answer_Sheet.csv", index=False)
print("Saved: DS5020_Walmart_Answer_Sheet.csv and note.txt")

,Task,Your value
0,P1,6435 rows; 45 stores; 143 weeks; 450 holiday s...
1,P2,P(S)=0.126496; P(H)=0.069930; P(Q)=0.125874; P...
2,P3,P(S∩H)=0.018959; P(S∩Q)=0.051748; P(H∩Q)=0.027...
3,B1,P(S|H)=0.271111; P(S|Hᶜ)=0.115622
4,B2,P(H|S) by Bayes=0.149877; counting=0.149877
5,B3,Highest P(S|group): Thanksgiving; largest P(gr...
6,B4,P(S|Q)=0.411111; P(Q|S)=0.409091
7,B5,"Holiday only: precision=0.271111, recall=0.149..."
8,B6,Twice as common: 0.465204; Half as common: 0.1...
9,N3,Profile a=0.793850558; Profile b=0.030069265


Saved: DS5020_Walmart_Answer_Sheet.csv and note.txt


## Submission checklist

- Keep this notebook and `Walmart.csv` together, then **Kernel → Restart & Run All**.
- Make sure the **Warm-up W1–W7** Markdown section is included. If the instructor literally requires paper, copy the hand calculations to paper.
- Include/export the completed **answer sheet** above and the **five-sentence `note.txt`**.
- All plots/tables and note use store numbers only. No addresses or managers' names.